# Integration tests de OP-06 `write_trips`

Notebook orientado a verificar el comportamiento integrado de la operación pública OP-06 `write_trips()`.

Estos tests no prueban helpers internos de forma aislada. En su lugar, preparan un `TripDataset` rico mediante operaciones públicas previas, lo validan formalmente y luego verifican que `write_trips()` materialice correctamente un artefacto formal de trips.

Objetivo:
- probar escritura formal de un `TripDataset` rico;
- verificar layout `.golondrina`;
- verificar sidecar `trips.metadata.json`;
- verificar metadata, evento y `OperationReport`;
- cubrir backend Parquet y backend Feather;
- probar fallas públicas relevantes de OP-06;
- no usar `read_trips`, porque eso pertenece a OP-07.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- resolución robusta del root del repositorio;
- imports generales;
- imports del módulo;
- helpers de testing reutilizables;
- carpeta local visible para artefactos.

### 0.1 Resolución del repositorio

Qué prepara: permite ejecutar el notebook desde distintas ubicaciones dentro del repositorio, agregando `src/` y el root al `sys.path` cuando corresponde.

Los artefactos de prueba se escribirán en una carpeta local relativa al directorio actual del notebook.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 
print("NOTEBOOK_ROOT =", NOTEBOOK_ROOT)
print("REPO_ROOT     =", REPO_ROOT)

NOTEBOOK_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips
REPO_ROOT     = C:\projects\pylondrina


### 0.2 Imports generales

Qué prepara: imports base, utilidades de filesystem, pandas y PyArrow para inspeccionar los artefactos físicos escritos por `write_trips()`.

In [2]:
import copy
import json
import shutil

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather

### 0.3 Imports del módulo

Qué prepara: operaciones públicas necesarias para construir una fixture rica y luego probar OP-06.

Importante: este notebook no importa `read_trips`, porque OP-07 tendrá su propio notebook.

In [3]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.datasets import TripDataset

from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips

from pylondrina.errors import ExportError, ValidationError

from pylondrina.io.trips import (
    write_trips,
    WriteTripsOptions,
)

### 0.4 Import del generador sintético

Qué prepara: acceso al generador usado para construir fixtures ricas de integración.

In [4]:
from scripts.synthetic_data.base_generator import generate_synthetic_trip_dataframe

### 0.5 Helpers de testing reutilizables

In [5]:
def show_ok(label: str):
    print(f"OK - {label}")


def get_issue_codes(issues):
    return [issue.code for issue in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente {code}. Codes actuales: {codes}"


def clone_tripdataset(trips: TripDataset) -> TripDataset:
    return copy.deepcopy(trips)


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def load_trips_sidecar(artifact_dir: Path) -> dict:
    sidecar_path = artifact_dir / "trips.metadata.json"
    assert sidecar_path.exists(), f"No existe sidecar: {sidecar_path}"
    return json.loads(sidecar_path.read_text(encoding="utf-8"))


def artifact_data_filename(storage_format: str) -> str:
    if storage_format == "parquet":
        return "trips.parquet"
    if storage_format == "feather":
        return "trips.feather"
    raise ValueError(f"storage_format no soportado: {storage_format!r}")


def artifact_data_file_path(artifact_dir: Path, storage_format: str) -> Path:
    return artifact_dir / artifact_data_filename(storage_format)


def artifact_total_size_bytes(artifact_dir: Path) -> int:
    return sum(p.stat().st_size for p in artifact_dir.rglob("*") if p.is_file())


def selected_categorical_columns(df: pd.DataFrame | None = None) -> list[str]:
    cols = [
        "mode",
        "purpose",
        "day_type",
        "time_period",
        "user_gender",
        "user_age_group",
        "income_quintile",
    ]
    if df is None:
        return cols
    return [c for c in cols if c in df.columns]


def series_as_string_with_na(series: pd.Series) -> pd.Series:
    return series.astype("string")

### 0.6 Configuración de display

In [6]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

show_ok("Imports y helpers cargados")

OK - Imports y helpers cargados


### 0.7 Carpeta visible de integración

Qué prepara: carpeta local para artefactos de persistencia generados por los tests.

La carpeta se crea en el directorio actual del notebook y se reinicia al ejecutar esta celda.

In [7]:
IT_ROOT = Path("./tmp_op06_write_trips_integration").resolve()


def reset_it_root() -> Path:
    if IT_ROOT.exists():
        shutil.rmtree(IT_ROOT)
    IT_ROOT.mkdir(parents=True, exist_ok=True)
    return IT_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = IT_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_it_root()
print("IT_ROOT =", root)
show_ok("Sección 0 lista")

IT_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips\tmp_op06_write_trips_integration
OK - Sección 0 lista


## Sección 1. Fixtures ricas reutilizables

Esta sección construye una fixture integrada realista usando el generador sintético, `import_trips_from_dataframe()` y `validate_trips()`.

La idea es que los tests de OP-06 no escriban un dataframe mínimo artificial, sino un `TripDataset` ya construido y validado mediante el flujo público previo del módulo.

### 1.1 Constantes de schema para la fixture rica

Qué prepara: campos requeridos, campos base y dominios canónicos suficientes para importación, validación y persistencia.

In [8]:
REQUIRED_FIELDS_ORDER = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_h3_index",
    "destination_h3_index",
    "origin_time_utc",
    "destination_time_utc",
    "trip_id",
    "movement_seq",
]

REQUIRED_FIELD_DTYPES = {
    "movement_id": "string",
    "user_id": "string",
    "origin_longitude": "float",
    "origin_latitude": "float",
    "destination_longitude": "float",
    "destination_latitude": "float",
    "origin_h3_index": "string",
    "destination_h3_index": "string",
    "origin_time_utc": "datetime",
    "destination_time_utc": "datetime",
    "trip_id": "string",
    "movement_seq": "int",
}

BASE_FIELD_DTYPES = {
    "origin_municipality": "string",
    "destination_municipality": "string",
    "timezone_offset_min": "int",
    "origin_time_local_hhmm": "string",
    "destination_time_local_hhmm": "string",
    "trip_weight": "float",
    "mode_sequence": "string",
    "mode": "categorical",
    "purpose": "categorical",
    "day_type": "categorical",
    "time_period": "categorical",
    "user_gender": "categorical",
    "user_age_group": "categorical",
    "income_quintile": "categorical",
}

CANONICAL_DOMAINS = {
    "mode": [
        "walk", "bicycle", "scooter", "motorcycle", "car",
        "taxi", "ride_hailing", "bus", "metro", "train", "other",
    ],
    "purpose": [
        "home", "work", "education", "shopping", "errand",
        "health", "leisure", "transfer", "other",
    ],
    "day_type": ["weekday", "weekend", "holiday"],
    "time_period": ["night", "morning", "midday", "afternoon", "evening"],
    "user_gender": ["female", "male", "other", "unknown"],
    "user_age_group": ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
    "income_quintile": ["1", "2", "3", "4", "5", "unknown"],
}

RICH_BASE_FIELDS = [
    "origin_municipality",
    "destination_municipality",
    "timezone_offset_min",
    "origin_time_local_hhmm",
    "destination_time_local_hhmm",
    "trip_weight",
    "mode_sequence",
    "mode",
    "purpose",
    "day_type",
    "time_period",
    "user_gender",
    "user_age_group",
    "income_quintile",
]

RICH_EXTRA_COLUMNS = [
    "activity_status",
    "education_level",
    "travel_time_bucket",
    "season",
    "fare_payment_type",
    "bike_lane_usage",
    "home_tenure",
]

### 1.2 Builder de schema rica

Qué prepara: un `TripSchema` suficientemente rico para importación y validación de datasets sintéticos realistas.

In [9]:
def make_field(name: str, dtype: str, *, required: bool = False, domain: DomainSpec | None = None) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=None,
        domain=domain,
    )


def make_rich_trip_schema() -> TripSchema:
    fields = {}

    for field_name in REQUIRED_FIELDS_ORDER:
        fields[field_name] = make_field(
            field_name,
            REQUIRED_FIELD_DTYPES[field_name],
            required=True,
        )

    for field_name, dtype_name in BASE_FIELD_DTYPES.items():
        domain = None
        if dtype_name == "categorical":
            domain = DomainSpec(values=CANONICAL_DOMAINS[field_name], extendable=True)

        fields[field_name] = make_field(
            field_name,
            dtype_name,
            required=False,
            domain=domain,
        )

    return TripSchema(
        version="1.1",
        fields=fields,
        required=list(REQUIRED_FIELDS_ORDER),
        semantic_rules=None,
    )

### 1.3 Builder de source dataframe rica

Qué prepara: un dataframe de entrada más rico que los smoke tests, usando generador sintético e incluyendo campos base y columnas extra.

In [10]:
def build_rich_source_dataframe(seed: int = 20260404, filas: int = 180) -> pd.DataFrame:
    df = generate_synthetic_trip_dataframe(
        filas=filas,
        seed=seed,
        duplicate_mode="none",
        tier_temporal="tier_1",
        tier1_datetime_format="utc_string_z",
        coord_format="numeric",
        h3_mode="provided_valid",
        trip_structure="multistage",
        max_movements_per_trip=3,
        base_fields=RICH_BASE_FIELDS,
        extra_value_domains={
            "mode": ["canon"],
            "purpose": ["canon"],
            "day_type": ["canon"],
            "time_period": ["canon"],
            "user_gender": ["canon"],
            "user_age_group": ["canon"],
            "income_quintile": ["canon"],
        },
        extra_columns=RICH_EXTRA_COLUMNS,
        null_ratio={
            "origin_municipality": 0.03,
            "destination_municipality": 0.03,
        },
    )
    return df

### 1.4 Fixtures base de integración

Qué prepara:
- `trip_schema_snapshot_small`;
- `tripdataset_canonical_small`;
- `tripdataset_unvalidated_small`;
- `tripdataset_validated_small`.

Aunque se llamen `small`, estas fixtures son suficientemente ricas para integración: pasan por importación pública, validación pública, metadata, schema efectivo, dominios y eventos previos.

In [11]:
trip_schema_snapshot_small = make_rich_trip_schema()

source_df_rich = build_rich_source_dataframe(filas=180)

tripdataset_canonical_small, canonical_import_report = import_trips_from_dataframe(
    source_df_rich,
    trip_schema_snapshot_small,
    source_name="synthetic_rich_trips",
    options=ImportOptions(
        keep_extra_fields=True,
        selected_fields=None,
        strict=False,
        strict_domains=False,
        single_stage=False,
        source_timezone=None,
    ),
    provenance={
        "source": {"name": "synthetic_generator", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
        "notes": ["fixture de integración OP-06 write_trips"],
    },
    h3_resolution=8,
)

assert canonical_import_report.ok is True
assert tripdataset_canonical_small.metadata["is_validated"] is False

tripdataset_unvalidated_small = clone_tripdataset(tripdataset_canonical_small)

tripdataset_validated_small = clone_tripdataset(tripdataset_canonical_small)
validated_report_fixture = validate_trips(
    tripdataset_validated_small,
    options=ValidationOptions(
        strict=False,
        validate_domains="full",
    ),
)

assert validated_report_fixture.ok is True
assert tripdataset_validated_small.metadata["is_validated"] is True

print("source_df_rich.shape =", source_df_rich.shape)
print("canonical shape     =", tripdataset_canonical_small.data.shape)
print("validated shape     =", tripdataset_validated_small.data.shape)
show_ok("Sección 1 lista")

source_df_rich.shape = (180, 33)
canonical shape     = (180, 33)
validated shape     = (180, 33)
OK - Sección 1 lista


### 1.5 Helper para construir fixtures grandes

Qué prepara: una fixture más grande para pruebas integradas sobre escritura física Feather. Se usa solo en el test de tamaño optimizado vs escritura naive.

In [12]:
def make_large_imported_trips_fixture(
    *,
    filas: int = 30_000,
    seed: int = 20260414,
) -> tuple[TripDataset, pd.DataFrame, object]:
    source_df = build_rich_source_dataframe(seed=seed, filas=filas)

    trips, import_report = import_trips_from_dataframe(
        source_df,
        trip_schema_snapshot_small,
        source_name="synthetic_rich_trips_large_for_feather",
        options=ImportOptions(
            keep_extra_fields=True,
            selected_fields=None,
            strict=False,
            strict_domains=False,
            single_stage=False,
            source_timezone=None,
        ),
        provenance={
            "source": {"name": "synthetic_generator", "entity": "trips"},
            "ingestion": {"created_at_utc": "2026-04-14T00:00:00Z"},
            "notes": ["fixture grande para integration tests de OP-06"],
        },
        h3_resolution=8,
    )

    assert import_report.ok is True
    return trips, source_df, import_report

## Sección 2. Integration tests de `write_trips()`

Esta sección prueba la operación pública OP-06 sin usar `read_trips`.

Los tests inspeccionan directamente los artefactos escritos en disco: archivo tabular, sidecar JSON, `OperationReport`, metadata en memoria y evento `write_trips`.

### Test 1 - write feliz con dataset rico y normalización de directorio

Qué prueba: caso principal correcto de `write_trips()` sobre una fixture validada y rica.

Verifica:
- escritura formal;
- normalización a `.golondrina`;
- archivo `trips.parquet`;
- sidecar `trips.metadata.json`;
- `OperationReport`;
- `summary`;
- `parameters`;
- evento en metadata;
- preservación de `trips.data`.

In [13]:
case_dir = make_case_dir("test_01_write_happy_normalized_parquet")

trips = clone_tripdataset(tripdataset_validated_small)
data_before = trips.data.copy(deep=True)
metadata_before = copy.deepcopy(trips.metadata)

base_path = case_dir / "sample_bundle"
expected_artifact_dir = case_dir / "sample_bundle.golondrina"

report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="overwrite",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

assert report.ok is True
assert expected_artifact_dir.exists()
assert expected_artifact_dir.is_dir()
assert (expected_artifact_dir / "trips.parquet").exists()
assert (expected_artifact_dir / "trips.metadata.json").exists()
assert not (expected_artifact_dir / "trips.feather").exists()

# Report observable
assert report.summary["n_rows"] == len(trips.data)
assert Path(report.summary["path"]) == expected_artifact_dir
assert report.summary["storage_format"] == "parquet"
assert report.summary["files_written"] == ["trips.parquet", "trips.metadata.json"]

assert report.parameters["storage_format"] == "parquet"
assert report.parameters["parquet_compression"] == "snappy"
assert report.parameters["normalize_artifact_dir"] is True
assert Path(report.parameters["path"]) == expected_artifact_dir

# Side effects en metadata del dataset
assert "dataset_id" in trips.metadata
assert "artifact_id" in trips.metadata
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == len(metadata_before["events"]) + 1
assert trips.metadata["events"][-1]["op"] == "write_trips"
assert trips.metadata["events"][-1]["parameters"] == report.parameters
assert trips.metadata["events"][-1]["summary"] == report.summary

# El dataframe no debe mutar
pd.testing.assert_frame_equal(trips.data, data_before)

# Sidecar consistente
sidecar = load_trips_sidecar(expected_artifact_dir)

assert sidecar["dataset_type"] == "trips"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"

assert sidecar["storage"]["format"] == "parquet"
assert sidecar["storage"]["options"]["compression"] == "snappy"

assert sidecar["files"]["data"] == "trips.parquet"
assert sidecar["files"]["metadata"] == "trips.metadata.json"

assert sidecar["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["artifact_id"] == trips.metadata["artifact_id"]

assert "schema" in sidecar
assert "schema_effective" in sidecar
assert "metadata" in sidecar
assert "provenance" in sidecar

assert sidecar["metadata"]["events"][-1]["op"] == "write_trips"
assert sidecar["metadata"]["events"][-1]["summary"] == report.summary

assert_json_safe(sidecar, "sidecar")
assert_json_safe(report.summary, "report.summary")
assert_json_safe(report.parameters, "report.parameters")
assert_json_safe(trips.metadata, "trips.metadata")

display(report.summary)
display(trips.data.head())
show_ok("Test 1 - write feliz Parquet con normalización de directorio")

{'n_rows': 180,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op06_write_trips_integration\\test_01_write_happy_normalized_parquet\\sample_bundle.golondrina',
 'dataset_id': 'tripds_8507fb594dc84162b822b91f810137fb',
 'artifact_id': 'art_536f8a84-aa40-4b82-96f5-09b5772ce3b3',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.300000,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c51b69fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.450000,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c57293fffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented


OK - Test 1 - write feliz Parquet con normalización de directorio


### Test 2 - write feliz y verificación de dictionary encoding en Parquet

Qué prueba: que `write_trips()` materializa columnas categóricas de forma eficiente en Parquet, usando la ruta pública completa y no helpers internos.

Este test inspecciona el archivo físico `trips.parquet` y verifica dictionary encoding en campos categóricos observables.

In [14]:
case_dir = make_case_dir("test_02_write_dictionary_encoding_parquet")

trips = clone_tripdataset(tripdataset_validated_small)
base_path = case_dir / "categorical_bundle"

report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

artifact_dir = case_dir / "categorical_bundle.golondrina"
parquet_path = artifact_dir / "trips.parquet"

assert report.ok is True
assert parquet_path.exists()

parquet_file = pq.ParquetFile(parquet_path)
checked_cols = []

try:
    names = parquet_file.schema_arrow.names

    for col_name in selected_categorical_columns(trips.data):
        if col_name in names:
            checked_cols.append(col_name)
            col_idx = names.index(col_name)

            encodings = {
                str(enc).upper()
                for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
            }

            assert any("DICTIONARY" in enc for enc in encodings), (
                f"{col_name} no quedó con dictionary encoding: {encodings}"
            )
finally:
    parquet_file.close()

assert checked_cols, "No se encontró ninguna columna categórica observable en el archivo Parquet."

display({"checked_cols": checked_cols})
show_ok("Test 2 - dictionary encoding en Parquet")

{'checked_cols': ['mode',
  'purpose',
  'day_type',
  'time_period',
  'user_gender',
  'user_age_group',
  'income_quintile']}

OK - Test 2 - dictionary encoding en Parquet


### Test 3 - write fatal por precondición de dataset no validado

Qué prueba: política clave de OP-06 con `require_validated=True`.

Si el dataset no está validado, la escritura debe abortar con `ValidationError` y no debe materializar el bundle.

In [15]:
case_dir = make_case_dir("test_03_write_fatal_unvalidated_parquet")

trips = clone_tripdataset(tripdataset_unvalidated_small)

raised = None
try:
    write_trips(
        trips,
        case_dir / "should_fail_bundle",
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format="parquet",
            parquet_compression="snappy",
            normalize_artifact_dir=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)
assert not (case_dir / "should_fail_bundle.golondrina").exists()

display(raised)
show_ok("Test 3 - write fatal por dataset no validado")

ValidationError(message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', details={'require_validated': True, 'validated_flag': False, 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op06_write_trips_integration\\test_03_write_fatal_unvalidated_parquet\\should_fail_bundle.golondrina', 'action': 'abort'}, issue=Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'require_validated': True, 'validated_flag': False, 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op06_write_trips_integration\\test_03_write_fatal_unvalidated_parquet\\should_fail_bundle.golondrina', 'action': 'abort'}), issues=(Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero 

OK - Test 3 - write fatal por dataset no validado


### Test 4 - write feliz con backend Feather y normalización de directorio

Qué prueba:
- escritura formal con backend Feather;
- normalización a `.golondrina`;
- existencia de `trips.feather` y sidecar;
- `OperationReport`;
- `summary`;
- `parameters`;
- side effects en metadata;
- sidecar backend-aware;
- preservación de `trips.data`.

In [16]:
case_dir = make_case_dir("test_04_write_happy_normalized_feather")

trips = clone_tripdataset(tripdataset_validated_small)
data_before = trips.data.copy(deep=True)
metadata_before = copy.deepcopy(trips.metadata)

base_path = case_dir / "sample_bundle"
expected_artifact_dir = case_dir / "sample_bundle.golondrina"

report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="overwrite",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=True,
    ),
)

assert report.ok is True
assert expected_artifact_dir.exists()
assert expected_artifact_dir.is_dir()
assert (expected_artifact_dir / "trips.feather").exists()
assert (expected_artifact_dir / "trips.metadata.json").exists()
assert not (expected_artifact_dir / "trips.parquet").exists()

# Report observable
assert report.summary["n_rows"] == len(trips.data)
assert Path(report.summary["path"]) == expected_artifact_dir
assert report.summary["storage_format"] == "feather"
assert report.summary["files_written"] == ["trips.feather", "trips.metadata.json"]

assert report.parameters["storage_format"] == "feather"
assert report.parameters["feather_compression"] == "lz4"
assert report.parameters["normalize_artifact_dir"] is True
assert Path(report.parameters["path"]) == expected_artifact_dir

# Side effects en metadata del dataset
assert "dataset_id" in trips.metadata
assert "artifact_id" in trips.metadata
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == len(metadata_before["events"]) + 1
assert trips.metadata["events"][-1]["op"] == "write_trips"
assert trips.metadata["events"][-1]["parameters"] == report.parameters
assert trips.metadata["events"][-1]["summary"] == report.summary

# Sidecar coherente
sidecar = load_trips_sidecar(expected_artifact_dir)

assert sidecar["storage"]["format"] == "feather"
assert sidecar["storage"]["options"]["compression"] == "lz4"
assert sidecar["storage"]["options"]["version"] == 2

assert sidecar["files"]["data"] == "trips.feather"
assert sidecar["files"]["metadata"] == "trips.metadata.json"

assert sidecar["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["artifact_id"] == trips.metadata["artifact_id"]
assert sidecar["metadata"]["events"][-1]["op"] == "write_trips"
assert sidecar["metadata"]["events"][-1]["summary"] == report.summary

# No muta trips.data
pd.testing.assert_frame_equal(
    trips.data.reset_index(drop=True),
    data_before.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

assert_json_safe(report.parameters, "report.parameters")
assert_json_safe(report.summary, "report.summary")
assert_json_safe(trips.metadata, "trips.metadata")
assert_json_safe(sidecar, "sidecar")

display(report.summary)
show_ok("Test 4 - write feliz Feather")

{'n_rows': 180,
 'files_written': ['trips.feather', 'trips.metadata.json'],
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op06_write_trips_integration\\test_04_write_happy_normalized_feather\\sample_bundle.golondrina',
 'dataset_id': 'tripds_8507fb594dc84162b822b91f810137fb',
 'artifact_id': 'art_82b572bd-6deb-4e05-a999-ebc750f4dd60',
 'dataset_id_status': 'preserved',
 'storage_format': 'feather'}

OK - Test 4 - write feliz Feather


### Test 5 - coherencia sidecar / backend / data file en Feather

Qué prueba:
- que el artefacto Feather sea autocontenible respecto del backend;
- que `storage.format` y `files.data` queden coherentes;
- que el archivo físico declarado por el sidecar exista.

Este test no usa `read_trips`: inspecciona directamente el sidecar y el layout en disco.

In [17]:
case_dir = make_case_dir("test_05_feather_sidecar_backend_coherence")

trips = clone_tripdataset(tripdataset_validated_small)
base_path = case_dir / "bundle"

write_report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=True,
    ),
)

artifact_dir = case_dir / "bundle.golondrina"
data_path = artifact_data_file_path(artifact_dir, "feather")
sidecar = load_trips_sidecar(artifact_dir)

assert write_report.ok is True
assert artifact_dir.exists()
assert data_path.exists()
assert data_path.name == "trips.feather"

assert sidecar["storage"]["format"] == "feather"
assert sidecar["storage"]["options"]["compression"] == "lz4"
assert sidecar["storage"]["options"]["version"] == 2
assert sidecar["files"]["data"] == "trips.feather"
assert sidecar["files"]["metadata"] == "trips.metadata.json"

assert write_report.summary["storage_format"] == "feather"
assert write_report.summary["files_written"] == ["trips.feather", "trips.metadata.json"]
assert write_report.summary["artifact_id"] == sidecar["artifact_id"]
assert write_report.summary["dataset_id"] == sidecar["dataset_id"]

display(sidecar["storage"])
display(write_report.summary)
show_ok("Test 5 - coherencia sidecar/backend/data file Feather")

{'format': 'feather', 'options': {'compression': 'lz4', 'version': 2}}

{'n_rows': 180,
 'files_written': ['trips.feather', 'trips.metadata.json'],
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op06_write_trips_integration\\test_05_feather_sidecar_backend_coherence\\bundle.golondrina',
 'dataset_id': 'tripds_8507fb594dc84162b822b91f810137fb',
 'artifact_id': 'art_aeec5c2d-d0f0-448b-ac0d-3a6ed1caa7df',
 'dataset_id_status': 'preserved',
 'storage_format': 'feather'}

OK - Test 5 - coherencia sidecar/backend/data file Feather


### Test 6 - compresión Feather inválida

Qué prueba:
- cierre del catálogo específico del backend Feather;
- una compresión no soportada debe abortar con `ExportError`;
- no se debe materializar el bundle.

In [18]:
case_dir = make_case_dir("test_06_invalid_feather_compression")

trips = clone_tripdataset(tripdataset_validated_small)

raised = None
try:
    write_trips(
        trips,
        case_dir / "invalid_compression_bundle",
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format="feather",
            feather_compression="snappy",  # inválida para este contrato
            normalize_artifact_dir=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert not (case_dir / "invalid_compression_bundle.golondrina").exists()

display(raised)
show_ok("Test 6 - compresión Feather inválida")

ExportError(message="La compresión Feather 'snappy' no está soportada para write_trips.", code='WRT.OPTIONS.UNSUPPORTED_FEATHER_COMPRESSION', details={'compression': 'snappy', 'supported_compressions': ['lz4', 'zstd', 'uncompressed', None], 'action': 'abort'}, issue=Issue(level='error', code='WRT.OPTIONS.UNSUPPORTED_FEATHER_COMPRESSION', message="La compresión Feather 'snappy' no está soportada para write_trips.", field=None, source_field=None, row_count=None, details={'compression': 'snappy', 'supported_compressions': ['lz4', 'zstd', 'uncompressed', None], 'action': 'abort'}), issues=(Issue(level='error', code='WRT.OPTIONS.UNSUPPORTED_FEATHER_COMPRESSION', message="La compresión Feather 'snappy' no está soportada para write_trips.", field=None, source_field=None, row_count=None, details={'compression': 'snappy', 'supported_compressions': ['lz4', 'zstd', 'uncompressed', None], 'action': 'abort'}),))

OK - Test 6 - compresión Feather inválida


### Test 7 - representación física categórica observable en Feather

Qué prueba:
- equivalente estructural al test de dictionary encoding en Parquet;
- verifica que columnas categóricas queden como `dictionary` en el schema Arrow del `.feather`;
- usa la operación pública `write_trips()`.

In [19]:
case_dir = make_case_dir("test_07_feather_arrow_dictionary_representation")

trips = clone_tripdataset(tripdataset_validated_small)

report = write_trips(
    trips,
    case_dir / "categorical_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=True,
    ),
)

artifact_dir = case_dir / "categorical_bundle.golondrina"
feather_path = artifact_dir / "trips.feather"

assert report.ok is True
assert feather_path.exists()

table = feather.read_table(feather_path)

checked_cols = []
for col_name in selected_categorical_columns(trips.data):
    if col_name in table.schema.names:
        checked_cols.append(col_name)
        assert pa.types.is_dictionary(table.schema.field(col_name).type), (
            f"{col_name} no quedó como dictionary en Feather: {table.schema.field(col_name).type}"
        )

assert checked_cols, "No se encontró ninguna columna categórica observable en el schema Arrow del archivo Feather."

display({"checked_cols": checked_cols})
display(table.schema)
show_ok("Test 7 - representación física categórica observable en Feather")

{'checked_cols': ['mode',
  'purpose',
  'day_type',
  'time_period',
  'user_gender',
  'user_age_group',
  'income_quintile']}

movement_id: string
user_id: string
trip_id: string
movement_seq: int64
origin_longitude: double
origin_latitude: double
destination_longitude: double
destination_latitude: double
origin_time_utc: timestamp[ns, tz=UTC]
destination_time_utc: timestamp[ns, tz=UTC]
origin_h3_index: string
destination_h3_index: string
origin_municipality: string
destination_municipality: string
timezone_offset_min: int64
origin_time_local_hhmm: string
destination_time_local_hhmm: string
trip_weight: double
mode_sequence: string
mode: dictionary<values=string, indices=int8, ordered=0>
purpose: dictionary<values=string, indices=int8, ordered=0>
day_type: dictionary<values=string, indices=int8, ordered=0>
time_period: dictionary<values=string, indices=int8, ordered=0>
user_gender: dictionary<values=string, indices=int8, ordered=0>
user_age_group: dictionary<values=string, indices=int8, ordered=0>
income_quintile: dictionary<values=string, indices=int8, ordered=0>
activity_status: string
education_level: strin

OK - Test 7 - representación física categórica observable en Feather


### Test 8 - tamaño en disco: ruta optimizada vs ruta naive en Feather

Qué prueba:
- que la preparación categórica usada por `write_trips()` reduzca el tamaño del `.feather`;
- se compara contra una escritura manual naive del mismo contenido lógico, forzando categóricos a `object/string`.

Nota:
- esta prueba puede tardar más que las anteriores;
- si el notebook se vuelve pesado, se puede bajar `filas` en `make_large_imported_trips_fixture(...)`.

In [20]:
case_dir = make_case_dir("test_08_feather_size_optimized_vs_naive")

trips_large, source_df_large, import_report_large = make_large_imported_trips_fixture(
    filas=30_000,
    seed=20260414,
)

# 1) Ruta optimizada: write_trips() público.
trips_large_opt = clone_tripdataset(trips_large)

opt_report = write_trips(
    trips_large_opt,
    case_dir / "optimized_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=False,   # No se fuerza validate sobre la fixture grande
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=True,
    ),
)

optimized_artifact_dir = case_dir / "optimized_bundle.golondrina"
optimized_data_path = optimized_artifact_dir / "trips.feather"

assert opt_report.ok is True
assert optimized_data_path.exists()

# 2) Ruta naive/manual: mismo contenido, pero forzando categóricos a string/object antes de escribir.
naive_artifact_dir = case_dir / "naive_bundle.golondrina"
naive_artifact_dir.mkdir(parents=True, exist_ok=True)
naive_data_path = naive_artifact_dir / "trips.feather"

df_naive = trips_large.data.copy(deep=True)
for col in selected_categorical_columns(df_naive):
    df_naive[col] = series_as_string_with_na(df_naive[col]).astype(object)

table_naive = pa.Table.from_pandas(df_naive, preserve_index=False)
feather.write_feather(
    table_naive,
    naive_data_path,
    compression="lz4",
    version=2,
)

assert naive_data_path.exists()

optimized_size = optimized_data_path.stat().st_size
naive_size = naive_data_path.stat().st_size
size_ratio = naive_size / optimized_size

assert optimized_size < naive_size, (
    f"Se esperaba que el archivo optimizado fuera más pequeño. "
    f"optimized={optimized_size}, naive={naive_size}"
)

display(
    pd.DataFrame(
        [
            {"variant": "optimized", "bytes": optimized_size},
            {"variant": "naive", "bytes": naive_size},
            {"variant": "naive_over_optimized_ratio", "bytes": size_ratio},
        ]
    )
)

show_ok("Test 8 - tamaño optimizado vs naive en Feather")

,variant,bytes
0,optimized,5.201058e+06
1,naive,6.299226e+06
2,naive_over_optimized_ratio,1.211143e+00


OK - Test 8 - tamaño optimizado vs naive en Feather


### Test 9 - escritura con `require_validated=False`

Qué prueba:
- `write_trips()` puede persistir un dataset no validado cuando el usuario desactiva explícitamente la precondición;
- la operación no certifica el dataset;
- el sidecar y la metadata preservan `is_validated=False`.

In [21]:
case_dir = make_case_dir("test_09_require_validated_false")

trips = clone_tripdataset(tripdataset_unvalidated_small)

report = write_trips(
    trips,
    case_dir / "unvalidated_allowed_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=False,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

artifact_dir = case_dir / "unvalidated_allowed_bundle.golondrina"

assert report.ok is True
assert artifact_dir.exists()
assert (artifact_dir / "trips.parquet").exists()
assert (artifact_dir / "trips.metadata.json").exists()

assert report.parameters["require_validated"] is False
assert trips.metadata["is_validated"] is False
assert trips.metadata["events"][-1]["op"] == "write_trips"

sidecar = load_trips_sidecar(artifact_dir)
assert sidecar["metadata"]["is_validated"] is False
assert sidecar["metadata"]["events"][-1]["op"] == "write_trips"

display(report.summary)
show_ok("Test 9 - write con require_validated=False")

{'n_rows': 180,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op06_write_trips_integration\\test_09_require_validated_false\\unvalidated_allowed_bundle.golondrina',
 'dataset_id': 'tripds_8507fb594dc84162b822b91f810137fb',
 'artifact_id': 'art_b0f7223c-bdbd-4283-9aff-c8e29791527c',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

OK - Test 9 - write con require_validated=False
